# Objectives
Welcome to the second Natural Language Processing lab, where we dive into the initial preprocessing stage of the NLP pipeline.

Today's objectives are:

1. 🧽 Practice using string operations and **regular expressions** as tools for initial data cleaning of text data.
2. 📋 Compare how different choices in **rules-based** tokenization and normalization affect results of the downstream task of sentiment analysis.
3. 🔢 Compare rules-based with **data-driven** tokenization and how it affects results of the downstream task.

## Packages
Required packages are listed in this lab's [requirements.txt](https://github.com/hertie-nlp-f2026/materials/blob/main/labs/02_session-2/requirements.txt). We will be focusing on three essential NLP packages today:

1. `re`: the Python built-in module for **regular expressions**.
    - [Python module](https://docs.python.org/3/library/re.html)
    - Cheat sheets: [CoderPad](https://coderpad.io/regular-expression-cheat-sheet/), [DataQuest](https://www.dataquest.io/cheat-sheet/regular-expressions-cheat-sheet/)
    - [Everything you need to know and then some!](https://www.regular-expressions.info/)
2. `spacy`: the current NLP standard for **rules-based** language processing pipeline.
    - [Linguistic Features](https://spacy.io/usage/linguistic-features)
    - [Processing Pipeline](https://spacy.io/usage/processing-pipelines)
3. `tokenizers`: from the HuggingFace ecosystem providing the **data-driven** tokenizers that enable the latest DL/LLMs.
    - [Subword Tokenization](https://huggingface.co/learn/llm-course/chapter2/4#subword-tokenization)
    - Training Algorithms: [BPE](https://huggingface.co/learn/llm-course/chapter6/5), [WordPiece](https://huggingface.co/learn/llm-course/chapter6/6), [Unigram](https://huggingface.co/learn/llm-course/chapter6/7)

Honourable mentions:

1. `nltk`: an older Natural Language Toolkit (NLTK) that provides finer-grained configuration of some preprocessing operations.
    - We use their [Snowball stemmer](https://www.nltk.org/api/nltk.stem.snowball.html) because spaCy doesn't provide one!
2. `sklearn`: we use scikit-learn for all of the vectorization and sentiment analysis classification
    - [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) actually wraps a lot of tokenization operations. Here, we use it to tokenize N-grams.

In [41]:
#| code-summary: 'Import packages'
#| output: false

# Data packages
import pandas as pd
pd.set_option('display.max_colwidth', None) #default: 50 chars
pd.set_option('display.max_columns', None) #default: 20 columns
import numpy as np

# NLP packages
import re
from sklearn.feature_extraction.text import CountVectorizer
import spacy
from nltk.stem.snowball import SnowballStemmer
from tokenizers import Tokenizer, pre_tokenizers
from tokenizers.models import BPE, WordPiece, Unigram
from tokenizers.trainers import BpeTrainer, WordPieceTrainer, UnigramTrainer

# Classification libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score, classification_report

# Data Cleaning Activity

Even before tokenization, we typically, especially for messy "in the wild" text data, need to do some data cleaning which can entail:

- Filtering in/out texts (including/excluding)
- Extracting information from texts
- Transforming texts: standardize, restructure, remove
  
We practice **transforming** and then **extracting** information using the (completely!) fictional data below.

## Text Input
A Hertie RA made a mistake on a Hertie alumni donation form and asked for information in open-ended format! It's up to us to clean and then structure the data as best we can. The form question is:

> Please write your name, email address, program, year of graduation, donor list hashtag (if any), and your alumni donation amount in Euros.

In [2]:
#| code-summary: "Load data cleaning dataset"
dfc = pd.read_csv('nlp_lab2_data_cleaning.csv')
dfc.head()

,text
0,"Hello, my name is Nikki Nikki Glaser at n.glaser@students.hertie-school.org, and I graduated from #HertieLoveeeee in 2025 in the M.D.S. program. Donating €200!"
1,Conan O'Brien (conan@tbs.com) here! I can proudly say that I am a proud mds graduate of 2021. #TeamCoco. Donating €2021.
2,Hey peeps! This is your Mds graduate 2023 Natasha Leggero showing \$ome #HertieLuv with a nice gift of €50. n.leggero@gmail.com.
3,"Dear #HertieLuuuv, I am Dave Chappelle Chappelle from MDS 2027 wishing you well. My donation will be €1000. Email: dave_chapelle@yahoo.com."


## Cleaning with String Replace
Let's just look at the first issue:

1. Correct all fuzzy spellings of `'#HertieLove'`

With simple **string operations**, we can only replace exact strings but that becomes unfeasible if the dataset were much larger.

In [3]:
#| code-summary: "Apply text corrections using regex"
#| code-fold: false
dfc['text2'] = dfc['text'].apply(lambda x: x.replace('HertieLoveeeee', 'HertieLove'))
dfc['text2'] = dfc['text2'].apply(lambda x: x.replace('HertieLuv', 'HertieLove'))
dfc['text2'] = dfc['text2'].apply(lambda x: x.replace('HertieLuuuv', 'HertieLove'))
dfc[['text2']].head()

,text2
0,"Hello, my name is Nikki Nikki Glaser at n.glaser@students.hertie-school.org, and I graduated from #HertieLove in 2025 in the M.D.S. program. Donating €200!"
1,Conan O'Brien (conan@tbs.com) here! I can proudly say that I am a proud mds graduate of 2021. #TeamCoco. Donating €2021.
2,Hey peeps! This is your Mds graduate 2023 Natasha Leggero showing \$ome #HertieLove with a nice gift of €50. n.leggero@gmail.com.
3,"Dear #HertieLove, I am Dave Chappelle Chappelle from MDS 2027 wishing you well. My donation will be €1000. Email: dave_chapelle@yahoo.com."


## Cleaning with Regex
This is where the utility of **regular expressions** (regex) comes in! We use regex to identify fuzzy or variable patterns and then substitute or extract text based on our needs.

A good process to follow for verifying regex's (it usually takes a few to many rounds!):

1. Create your regex pattern 
2. Test on a representative string(s) -> **Great tool for this is [regex101](https://regex101.com/)**
3. Apply to an entire list of strings (here, in a Pandas series)
4. Check for correctness

In [ ]:
#| code-summary: "MODIFY: Test regex pattern on individual strings"
#| code-fold: false
text = '#HertieLoveeeee'
pattern = r"#HertieLove" # MODIFY to find the pattern variations to replace
repl = '#HertieLove'
re.sub(pattern, repl, text)

'#HertieLoveeeee'

In [ ]:
#| code-summary: "MODIFY: Apply to all strings in dataset"
dfc['text2'] = dfc['text'].apply(lambda x: re.sub(r"#HertieLove", '#HertieLuv', '#HertieLuuuv', x)) #1. MODIFY for '#HertieLove'
dfc[['text2']].head()

TypeError: sub() missing 1 required positional argument: 'string'

## More Cleaning with Regex
Let's continue our cleaning list with regex:

1. Correct all fuzzy spellings of `'#HertieLove'` **DONE**
2. Standardize all versions of MDS to `'MDS'`.
3. Correct invalid starting years to `'ERROR'` (only for 202x)
4. Correct accidentally repeated names (leave only one)

In [ ]:
#| code-summary: "MODIFY: Apply text corrections using regex"    #Write Regex code by replacing things
dfc['text2'] = dfc['text'].apply(lambda x: re.sub(r"#HertieLove", '#HertieLove', x)) #1. MODIFY for '#HertieLove'
dfc['text2'] = dfc['text2'].apply(lambda x: re.sub(r"[MDS]", 'M.D.S.', 'Mds', x)) #2. MODIFY for 'MDS'
dfc['text2'] = dfc['text2'].apply(lambda x: re.sub(r"ERROR", r"ERROR", x)) #3. MODIFY for 'ERROR' years (202x)
#dfc['text2'] = dfc['text2'].apply(lambda x: re.sub(r"(.*)\1", r"\1", x)) #4. MODIFY for repeated names
dfc[['text2']].head()

TypeError: 'str' object cannot be interpreted as an integer

## Extracting with Regex

Extracting information using regex works similarly except the found string needs to be saved to a data structure.

1. Full name (assuming consecutive capitalized title case)
2. Email address
3. Donation amount

Here, we collect lists of information and then arrange them in a Pandas `DataFrame`.

In [ ]:
#| code-summary: "MODIFY: Extract text strings using regex"
full_names = dfc['text2'].apply(lambda x: re.search(r"\w", x).group()) #1. MODIFY for full names
emails = dfc['text2'].apply(lambda x: re.search(r"\w", x).group()) #2. MODIFY for email addresses
donation_amounts = dfc['text2'].apply(lambda x: re.search(r"\d", x).group(1)) #3. MODIFY for donation amounts (capture group 1)
pd.DataFrame({'Full Name': full_names, 'Email': emails, 'Donation': donation_amounts})

# Research Question
We now move onto a full pipeline research question. Our mini-research question for this lab is:

>  **How do preprocessing choices in tokenization and normalization affect a downstream task such as sentiment analysis?**

![pipeline](images/NLP_Lab2_pipeline.png)

# Text Input
We'll be using a Google Maps restaurant reviews dataset from Türkiye 🇹🇷 found on [Kaggle](https://www.kaggle.com/datasets/denizbilginn/google-maps-restaurant-reviews). It has been modified for positive and negative sentiment classes (originally from a 5-point scale, neutral "3" dropped).

In [17]:
#| code-summary: "Load modified restaurant reviews dataset"
df = pd.read_csv('reviews_sentiment.csv')
print(f'The shape of this dataset is {df.shape}')
df.sample(2)

The shape of this dataset is (928, 2)


,text,sentiment
855,Really delicious burger. I would recommend it to everyone. Thanks to the smiling staff.,positive
165,Service is very slow. The ayran comes in advance; neither foam remains nor we saw here for the first time that buttermilk is paid.,negative


# Data Cleaning
Luckily, these reviews are fairly clean. The one issue is that the way the Turkish currency of the **lira** is used is not consistent. Let's standardize `'tl'`, `'TL'`, and `'liras'` all to `'lira'`.

In [18]:
#| code-summary: "MODIFY: Standardize 'lira' using regex"
texts = [
    "46 liras for 6 meatballs; 30 minutes or more waiting time.",
    'It is very interesting that french fries are 114 TL.',
    'We ate cafe Inn pizza. It was 70 tl.',
    'In general; I was satisfied; since the hamburger is a little small;'] # Don't change the 'tl' in 'little'!

pattern = r"lira" # MODIFY for finding alternative variations for 'lira'
repl = 'lira'
for t in texts:
    print(re.sub(pattern, repl, t))

46 liras for 6 meatballs; 30 minutes or more waiting time.
It is very interesting that french fries are 114 TL.
We ate cafe Inn pizza. It was 70 tl.
In general; I was satisfied; since the hamburger is a little small;


In [19]:
#| code-summary: "CHECK: Apply to the dataset and check"
df1 = df.copy()
df1['text'] = df1['text'].apply(lambda x: re.sub(pattern, repl, x))
df1[df1['text'].str.contains(r"\d+[Tt][Ll]")]

,text,sentiment
58,The breakfast was very good; I recommend it; the presentations were also eye-catching. We paid 240tl for the table you see in the photo; including 6 teas and 2 Turkish coffees.,positive
126,A place where you pay for the view. In general; although the breakfast is good; 80tl per person for two people 160tl is a bit high.,positive
222,I recommend the handmade hamburger at a good price; reasonable (50TL) and satisfying. Sprinkled village breakfast (125 TL per person) is a weak; ordinary breakfast.,positive
439,Tantuni is delicious; service is fast; service is quality. Price of one portion is 85TL,positive
491,It's nice that it's open every time in a day. Turkish coffee + tea + 2 pastries =52TL. Service is average... Turkish coffee is not tasty.,positive
534,10 mussels + half a kokorec is 240TL. Also; the place smells like toilet.,negative
568,Prices too high Turkish coffee 42.5TL Salty cookies weak.,positive
700,We ordered cafe de paris; it was delicious; and the price was 35tl.,positive


In [20]:
#| code-summary: "MODIFY: An edge case to fix?"
texts = ['We paid 240tl for the table you see in the photo;',
         'Prices too high Turkish coffee 42.5TL Salty cookies weak.']

pattern = r"(\d)" # MODIFY for alternative variations for 'lira'
repl = r"\1 lira"
for t in texts:
    print(re.sub(pattern, repl, t))

We paid 2 lira4 lira0 liratl for the table you see in the photo;
Prices too high Turkish coffee 4 lira2 lira.5 liraTL Salty cookies weak.


In [30]:
#| code-summary: "CHECK: Apply to the dataset and check"
df2 = df1.copy()
df2['text'] = df2['text'].apply(lambda x: re.sub(pattern, repl, x))
df2[df2['text'].str.contains(r"\d+[Tt][Ll]")]

,text,sentiment


# Rules-Based Tokenization

After data cleaning, tokenization is the step of splitting up each text into smaller parts of text, called **tokens**. These tokens are then represented numerically and then this representation is used to enable the downstream NLP task(s). There are generally two branches of tokenization:

1. **Rules-based:** We make choices based on rules of each language and the needs of the research or application domain.
2. **Data-driven:** The tokenization is driven by statistical algorithms.

## Rules-Based: Tokenize on Whitespace
We can use Python's built-in string `split()` method to simply split each text into tokens by whitespace. This can be our baseline for our downstream task. As for each our tokenization step, we first see how the tokenization looks on one example text, and then apply it to the dataset.

In [31]:
#| code-summary: "Split tokenize on example text"
#| code-fold: false
text = "The San Francisco-based restaurant doesn't charge $10."
tokens = text.split()
tokens

['The', 'San', 'Francisco-based', 'restaurant', "doesn't", 'charge', '$10.']

In [32]:
#| code-summary: "Split tokenize on dataset"
#| code-fold: false
dft = df2.copy() #for tokenization
dft['tokens'] = dft['text'].str.split() #entire dataset
dft[['text', 'tokens']].sample(2)

,text,tokens
729,I tried peking duck with orange and soy sauce. It had a somewhat offensive odor. Good in terms of taste.,"[I, tried, peking, duck, with, orange, and, soy, sauce., It, had, a, somewhat, offensive, odor., Good, in, terms, of, taste.]"
90,Every time I come to Fethiye; I stop by and fill my stomach with pleasure.,"[Every, time, I, come, to, Fethiye;, I, stop, by, and, fill, my, stomach, with, pleasure.]"


### View Count Vector
We use the "count vector" representation provided by scikit-learn's `'CountVectorizer'` method.

In [33]:
#| code-summary: "Define function for count vectorizer"
def count_vectorize_to_df(s_tokens, s_text):
    '''Vectorize each text into a count vector of token frequencies, and output as DataFrame.'''
    vectorizer = CountVectorizer(
        analyzer=lambda tokens: tokens, # disable CountVectorizer's tokenization
    )
    X_bow = vectorizer.fit_transform(s_tokens)
    print(f'The vocabulary size is {len(vectorizer.vocabulary_)} tokens')
    return pd.DataFrame(X_bow.toarray(), index=s_text, columns=vectorizer.get_feature_names_out())

In [34]:
#| code-summary: "Vectorize: view vocabulary size and count vector"
#| code-fold: false
df_bow = count_vectorize_to_df(dft['tokens'], dft['text'])
df_bow.sample(1)

The vocabulary size is 3440 tokens


,!,"""Would","""dove""",&,(1,(5,(7,(Alçatı;,(Halal,(depending,(in,(not,(tavla).,+,.,.Bad,/,0,1,2,3,4,5,6,7,8,9,:,:2,=5,A,A-MA-ZING!,Abidin,Absolutely,Abundant;,Actually;,Adana,Adana;,Affordable,After,Akyaka.,Alanya,Alanya.,Alcohol,Alfredo;,Ali,All,Along,Alsancak,Also,"Also,",Also;,Although,Amazing,Amazing!,Ambiance,Ambience,Ambient,An,And,Ankara's,Ankara;,Antakya,Antalya.,Antep,Antep;,Apart,Appetizers,Arab,Arabic,Arabs,Armut.,As,Asian,At,Atakum,Atmosphere,Atom,Attention,Authentic,Average,Avocado,Awesome,Awesomeee,Ayran,Ayvalik,Ayvalık,Ayvalık.,Ayvalık;,Babas;,Bad,Bagel,Be,Beach.,Beautiful,Beautiful;,Beautifully,Beef,Begendi,Being,Benedict,Bentos,Berry,Best,Beylikduzu.,Beyran,Beyran.,Beyrani,Beyti.,Big,Bitter;,Black,Blueberries.,Bodrum,Bodrum.,Boiled,Bon,Bosphorus,Bostanlı;,Both,Boyoz,Boyozes,Boza,Boza's,Bozacisi,Bozacısı,Bread,Bread;,Breakfast,Brilliant,Bulgogi,Bunbun,Burger,Burger.,Burgers,Bursa.,Business,But,By,COME.,Cacik,Cafe,Cakalli,Cappadocia.,Castle,Cheese,Cheesecake,Cheesecakes,Chefs,Chicken,Chinese,Chocolate,Cici,Cizlama,Classic,Classical,Clean,Clean.,Cola.,Coming,Compared,Congratulations,Congratulations;,Contrary,Cooking,Covered,Cream,Cream.,Cream;,Credit,Crispy,Croissant.,Crowded,Customer,Cyprus,DEFINITELY,DO,Datca.,Definitely,Definitely.,Delicious,Delicious!,Delicious.,Deliciousss!,Depending,Deserts,Despite,Desserts,Diavola,Different,Dining,Dishes,Diyarbakır,Do,Don't,Doner,Dry,Due,During,Each,Eating,Egg,Eggs,Eggs;,Ekmek,Elite;,Employees,Especially,Etli,Etliekmek,Even,Every,Everyone,Everything,Excellent,Excellent.,Except,Expensive,Eye,Famous,Fantastic,Fast,Fast;,Fellas.,Fethiye,Fethiye.,Fethiye;,Few,Fikret's,Finding,First,Fish,Flat,Flavor,Folk,Food,Food/desserts,For,Frankly;,Friday,Friendly,GF,GOAT,Gaziantep,Gaziantep!,Gaziantep;,Generally,Good,Good.,Great,Great.,Greece,Half,Hamburger,Hatay,Have,He's,Here,High,Highly,Home,Homemade,Hookah,Hours,However;,Hünkar,I,I'll,I'm,I've,Ice,If,In,Including,Incredibly,Inegöl,Inn,Inside,Interest,Interior,Iskender,Isn't,Istanbul,Istanbul-based,Istanbul.,It,It's,Italian,Italians,Its,Izmir,Izmir;,I’ve,July,June,Just,KFC.,Karaköy,Karaköy.,Kaş,Kaş.,Kaş;,Kebab,Kebab.,Kebabs,Kimbap,Kofteci,Kokorec,Koksal;,Konya,Kordon..,Korea,Korea.,Korean,Kos;,Koycegiz,Kukis,Kumpir,Kumru,Kumru's,Köyceğiz,Kızılay,Labor,Lahmacun,Lahmacun.,Legend,Legendary,Lemonade,Liked,Liked.,Limited,Lipa,Liver,Local,Located,Location,Lokum,Lots,Louvre,Louvre;,Love,Loved.,Lovely,Lovely;,Mac,Mado...,Magnificent,Magro,Magro.,Many,Margarita,Marmaris,Marmaris.,Marmaris;,Master,Maybe,Meals,Meat,Meatballs,Meats,Meaty,Menemen,Menemen;,Menu,Meram,Mersin,Metanet's,Mexican,Mia,Mixed,Moblan,Mohair,Moreover;,Most,Mr.,Mumbar,Mushroom,Music;,Muğla.,My,NOT,Naturally;,Nazik,Nice,No,Noodles.,Not,Nothing,Nutella,Ocakbasi,Of,Offering,Oh,Oh;,On,Once,One,Only,Orange,Order,Ordered,Orders,Other,Our,Overall,Overcrowded,Paris.,Parmesan,Pasta,Pastries,Patisserie,Pavlova,People,Perfect,Perfect!,Perfect;,Personally;,Pie,Pistachio,Pizza,Places.,Plain,Please,Plus,Poached,Popcorn,Popular,Portions,Pose,Presentation;,Pretty,Price,Prices,Pukka,Quality,Quite,Raisins,Raki,Real,Really,Reasonable,Recommend,Recommend!,Recommend.,Recommended,Recommended.,Renovated,Restaurant,Reviews,Rib,Rice,Rice;,Right,Roast,Roasted,Roasting,Roman,Rome).,Rotary,Sakarası,Salad,Salad;,Salty,Samet;,Samsun,Samsun.,Samsun;,Sandwich's,Sariyer.,Sarıyer.,Sausage,Scrambled,Sea,Sea.,Seaside,Seaside;,Seasonal,Seriously,Service,"Service,",Service;,She,Shrimp,Simplicity,Since,Slice,Small,So,Some,Something,Souffle,Souffles,Soundproofing,Soup,Soups,Spacious;,Sprinkled,Staff,Starbucks..,Started,Still;,Stones,Strawberry,Stuffed,Stylish;,Suflor,Summer;,Sunday,Super,Sushi,SushiCo,TL,TL).,TL.,TL;,Taco;,Tantuni,Tantuni!,Taste,Taste;,Tasty,Tasty;,Tea,Team!,Teas,Terrible,Terrible.,Testaccio,Texts,Thai,Thank,Thanks,Thanks.,That's,The,Their,Then,There,There's,Therefore;,They,They're,They've,Thin,This,Those,Three,Toast.,Toilets,Tom,Too,"Trabzon,",Traditional,Treats,Tried,Tripple,Trucks,Turkey's,Turkey

### Evaluate Sentiment Classifier
We use scikit-learn's `LogisticRegression` with a basic train/test split and cross-validation process to evaluate our preprocessing throughout the remainder of this lab. We look at the F1 scores (harmonic mean of precision and recall) for 

In [35]:
#| code-summary: "Define function for sentiment analysis classifier"
def sentiment_analysis_pipeline(X, y, min_df=1, max_df=1.0, max_features=None, ngram_range=(1,1)):
    '''docstring'''

    # For uni-gram vs. bi-grams
    if ngram_range == (1,1):
        analyzer = (lambda tokens: tokens)
    else:
        analyzer = 'word'
        X = X.apply(' ' .join) #bi-grams need full strings
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2, #80/20 split
        random_state=42,
        stratify=y
    )
    # Vector representation (train/test separately!)    
    vectorizer = CountVectorizer(
        analyzer=analyzer,
        min_df=min_df, max_df=max_df, max_features=max_features, ngram_range=ngram_range # allow user setting
    )
    X_train_bow = vectorizer.fit_transform(X_train)
    X_test_bow = vectorizer.transform(X_test)
    print(f'The vocabulary size is {len(vectorizer.vocabulary_)} tokens')
    
    # Find best parameter on training data
    f1_scorer = make_scorer(f1_score, pos_label='positive')
    grid = GridSearchCV(
        LogisticRegression(max_iter=1000),
        param_grid={'C': [0.01, 0.1, 1, 10, 100]},
        cv=5,
        scoring=f1_scorer
    )
    grid.fit(X_train_bow, y_train)
    best_model = grid.best_estimator_
    
    # Evaluate on held-out test data
    y_pred = best_model.predict(X_test_bow)
    print(classification_report(y_test, y_pred))

In [36]:
#| code-summary: "Run classifier on string split tokenization"
#| code-fold: false
sentiment_analysis_pipeline(dft['tokens'], dft['sentiment'])

The vocabulary size is 3033 tokens
              precision    recall  f1-score   support

    negative       0.73      0.37      0.49        30
    positive       0.89      0.97      0.93       156

    accuracy                           0.88       186
   macro avg       0.81      0.67      0.71       186
weighted avg       0.86      0.88      0.86       186



## Rules-Based: Tokenize with spaCy
We now turn to more sophisticated ways to tokenize based on linguistic properties provided by the spaCy package.

In [43]:
#| code-summary: "spaCy tokenize on example text"
#| code-fold: false
spacy.cli.download('en_core_web_sm') #Download (once) if you haven't yet # sm(all) is faster but no embeddings
nlp = spacy.load('en_core_web_sm')
text = "The San Francisco-based restaurant doesn't charge $10."
doc = nlp(text)
print([token.text for token in doc])

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
['The', 'San', 'Francisco', '-', 'based', 'restaurant', 'does', "n't", 'charge', '$', '10', '.']


In [44]:
#| code-summary: "spaCy tokenize on dataset"
#| code-fold: false
docs_as_tokens = list(nlp.pipe(dft['text'])) # entire dataset
dft['tokens_spacy'] = [
    [token.text for token in doc] # SAME baseline tokenized text, no normalization
    for doc in docs_as_tokens] # per-document
dft[['text', 'tokens_spacy']].sample(2)

,text,tokens_spacy
351,I definitely come here every time I come; I like both breakfast and evening meals; but prices have increased a lot from last year to this year.,"[I, definitely, come, here, every, time, I, come, ;, I, like, both, breakfast, and, evening, meals, ;, but, prices, have, increased, a, lot, from, last, year, to, this, year, .]"
66,Seaside great view!,"[Seaside, great, view, !]"


In [45]:
#| code-summary: "Vectorize: view vocabulary size and count vector"
df_bow = count_vectorize_to_df(dft['tokens_spacy'], dft['text'])
df_bow.sample(1)

The vocabulary size is 2450 tokens


,,!,"""",&,'d,'ll,'m,'re,'s,'ve,(,),+,",",-,.,..,...,.Bad,/,0,1,2,3,4,5,6,7,8,9,:,;,=,?,A,Abidin,Absolutely,Abundant,Actually,Adana,Affordable,After,Akyaka,Alanya,Alcohol,Alfredo,Ali,All,Along,Alsancak,Also,Although,Alçatı,Amazing,Ambiance,Ambience,Ambient,An,And,Ankara,Antakya,Antalya,Antep,Apart,Appetizers,Arab,Arabic,Arabs,Armut,As,Asian,At,Atakum,Atmosphere,Atom,Attention,Authentic,Average,Avocado,Awesome,Awesomeee,Ayran,Ayvalik,Ayvalık,Babas,Bad,Bagel,Be,Beach,Beautiful,Beautifully,Beef,Begendi,Being,Benedict,Bentos,Berry,Best,Beylikduzu,Beyran,Beyrani,Beyti,Big,Bitter,Black,Blueberries,Bodrum,Boiled,Bon,Bosphorus,Bostanlı,Both,Boyoz,Boyozes,Boza,Bozacisi,Bozacısı,Bread,Breakfast,Brilliant,Bulgogi,Bunbun,Burger,Burgers,Bursa,Business,But,By,COME,Cacik,Cafe,Cakalli,Cappadocia,Castle,Cheese,Cheesecake,Cheesecakes,Chefs,Chicken,Chinese,Chocolate,Cici,Cizlama,Classic,Classical,Clean,Cola,Coming,Compared,Congratulations,Contrary,Cooking,Covered,Cream,Credit,Crispy,Croissant,Crowded,Customer,Cyprus,DEFINITELY,DO,Datca,Definitely,Delicious,Deliciousss,Depending,Deserts,Despite,Desserts,Diavola,Different,Dining,Dishes,Diyarbakır,Do,Doner,Dry,Due,During,Each,Eating,Egg,Eggs,Ekmek,Elite,Employees,Especially,Etli,Etliekmek,Even,Every,Everyone,Everything,Excellent,Except,Expensive,Eye,Famous,Fantastic,Fast,Fellas,Fethiye,Few,Fikret,Finding,First,Fish,Flat,Flavor,Folk,Food,For,Frankly,Friday,Friendly,GF,GOAT,Gaziantep,Generally,Good,Great,Greece,Halal,Half,Hamburger,Hatay,Have,He,Here,High,Highly,Home,Homemade,Hookah,Hours,However,Hünkar,I,Ice,If,In,Including,Incredibly,Inegöl,Inn,Inside,Interest,Interior,Is,Iskender,Istanbul,It,Italian,Italians,Its,Izmir,July,June,Just,KFC,Karaköy,Kaş,Kebab,Kebabs,Kimbap,Kofteci,Kokorec,Koksal,Konya,Kordon,Korea,Korean,Kos,Koycegiz,Kukis,Kumpir,Kumru,Kuymak,Köyceğiz,Kızılay,Labor,Lahmacun,Legend,Legendary,Lemonade,Liked,Limited,Lipa,Liver,Local,Located,Location,Lokum,Lots,Louvre,Love,Loved,Lovely,MA,Mac,Mado,Magnificent,Magro,Many,Margarita,Marmaris,Master,Maybe,Meals,Meat,Meatballs,Meats,Meaty,Menemen,Menu,Meram,Mersin,Metanet,Mexican,Mia,Mixed,Moblan,Mohair,Moreover,Most,Mr.,Mumbar,Mushroom,Music,Muğla,My,NOT,Naturally,Nazik,Nice,No,Noodles,Not,Nothing,Nutella,Ocakbasi,Of,Offering,Oh,On,Once,One,Only,Orange,Order,Ordered,Orders,Other,Our,Overall,Overcrowded,Paris,Parmesan,Pasta,Pastries,Patisserie,Pavlova,People,Perfect,Personally,Pie,Pistachio,Pizza,Places,Plain,Please,Plus,Poached,Popcorn,Popular,Portions,Pose,Presentation,Pretty,Price,Prices,Pukka,Quality,Quite,Raisins,Raki,Real,Really,Reasonable,Recommend,Recommended,Renovated,Restaurant,Reviews,Rib,Rice,Right,Roast,Roasted,Roasting,Roman,Rome,Rotary,Sakarası,Salad,Salty,Samet,Samsun,Sandwich,Sariyer,Sarıyer,Sausage,Scrambled,Sea,Seaside,Seasonal,Seriously,Service,She,Shrimp,Simplicity,Since,Slice,Small,So,Some,Something,Souffle,Souffles,Soundproofing,Soup,Soups,Spacious,Sprinkled,Staff,Starbucks,Started,Still,Stones,Strawberry,Stuffed,Stylish,Suflor,Summer,Sunday,Super,Sushi,SushiCo,TL,Taco,Tantuni,Taste,Tasty,Tea,Team,Teas,Terrible,Testaccio,Texts,Thai,Thank,Thanks,That,The,Their,Then,There,Therefore,They,Thin,This,Those,Three,Toast,Toilets,Tom,Too,Trabzon,Traditional,Treats,Tried,Tripple,Trucks,Turkey,Turkish,Two,Unfortunately,Unique,Vefa,Vegetables,Very,View,Waffles,Waiters,Way,We,Well,What,Whatever,When,Winter,With,Without,Wonderful,Worth,Would,Yesemek,You,Yum,Yummy,Yusuf,ZING,a,able,about,above,abroad,absolutely,accept,acceptable,accepted,access,accessible,accidentally,accompanied,accompanying,accordance,according,account,across,actually,ad,adana,adapted,add,added,addicted,addition,address,adds,adequate,adjacent,adjika,adult,advance,advantage,advise,afford,affordable,after,afternoon,afterward,again,against,age,ago,air,airport,airy,ala,alavara,alcohol,all,almond,almost,alone,already,also,although,always,am,amazed,amazement,amazing,ambiance,ambience,among,amount,an,anchovies,and,another,any,anything,anyway,anywhere,apart,apologies,appeal,ap

In [46]:
#| code-summary: "Run classifier on regex tokenization"
sentiment_analysis_pipeline(dft['tokens_spacy'], dft['sentiment'])

The vocabulary size is 2189 tokens
              precision    recall  f1-score   support

    negative       0.62      0.33      0.43        30
    positive       0.88      0.96      0.92       156

    accuracy                           0.86       186
   macro avg       0.75      0.65      0.68       186
weighted avg       0.84      0.86      0.84       186



## Rules-Based: Normalization
After tokenizing, there are numerous normalization choices to make including:

- **Case folding:** using `token.lower_` provided by spaCy's tokenizer
- **Stemming:** using NLTK's `SnowballStemmer` ("stems" may not result in real words)
- **Lemmatization:** using `token.lemma` provided by spaCy's lemmatizer (standardize to base meaning "lemma")
- **Remove non-alphabetical** using `token.is_alpha` provided by spaCy's tokenizer
- **Remove stopwords** using `token.is_stop` provided by spaCy's tokenizer and English stop word list
    - Stopword lists for different languages are available on spaCy's GitHub:
        - [English](https://github.com/explosion/spaCy/blob/master/spacy/lang/en/stop_words.py)
        - [German](https://github.com/explosion/spaCy/blob/master/spacy/lang/de/stop_words.py)
        - [Spanish](https://github.com/explosion/spaCy/blob/master/spacy/lang/es/stop_words.py)
        - [Hindi](https://github.com/explosion/spaCy/blob/master/spacy/lang/hi/stop_words.py)

In [47]:
#| code-summary: "OBSERVE: Test normalization choices on example text"
#| code-fold: false
text = "The San Francisco-based restaurant doesn't charge $10."
doc = nlp(text) # spaCy
my_stemmer = SnowballStemmer('english') # NLTK

# Normalization CHOICES
print([token.lower_ for token in doc]) # Case folding
print([my_stemmer.stem(token.text) for token in doc]) # Stemming
print([token.lemma_ for token in doc]) # Lemmaitzation -> case-fold with .lower()
print([token.text for token in doc if token.is_alpha]) # Remove non-alphabetical
print([token.text for token in doc if not token.is_stop]) # Remove stopwords

['the', 'san', 'francisco', '-', 'based', 'restaurant', 'does', "n't", 'charge', '$', '10', '.']
['the', 'san', 'francisco', '-', 'base', 'restaur', 'doe', "n't", 'charg', '$', '10', '.']
['the', 'San', 'Francisco', '-', 'base', 'restaurant', 'do', 'not', 'charge', '$', '10', '.']
['The', 'San', 'Francisco', 'based', 'restaurant', 'does', 'charge']
['San', 'Francisco', '-', 'based', 'restaurant', 'charge', '$', '10', '.']


### Normalizaiton Choices
We apply the normalization choices to the entire dataset using a **nested list comprehension**.
Try various options and combinations and see how the various classification metrics change. Some examples are provided.

In [63]:
#| code-summary: "MODIFY: Make normalization choices and evaluate classifier"
#| code-fold: false
docs_as_tokens = list(nlp.pipe(dft['text'])) # entire dataset
dft['tokens_spacy'] = [

    # MODIFY HERE
    #[token.text for token in doc], # no normalization)
    [token.lemma_.lower() for token in doc] # combine lemma + lower
    #[token.lower for token in doc if token.is_alpha] # combine case-folding, removal of stopwords
    #[token.lower for token in doc if token.is_alpha and not token.is_stop] # combine case-folding, removal of non-alpha/stopwords
    for doc in docs_as_tokens] # per-document

# Vectorize (vocab count), run classifier
sentiment_analysis_pipeline(dft['tokens_spacy'], dft['sentiment'])

The vocabulary size is 1643 tokens
              precision    recall  f1-score   support

    negative       0.85      0.37      0.51        30
    positive       0.89      0.99      0.94       156

    accuracy                           0.89       186
   macro avg       0.87      0.68      0.72       186
weighted avg       0.88      0.89      0.87       186



## Rules-Based: N-grams
N-grams combine N neighboring words into one token using scikit-learn's `CountVectorizer`. It may help to limit the number of features using `max_features` as the number of N-gram tokens can become very large.

In [65]:
#| code-summary: "MODIFY: Choose N-grams options, vectorize, and evaluate classifier"
#| code-fold: false
sentiment_analysis_pipeline(
    dft['tokens_spacy'], dft['sentiment'],
    max_features=5000, # MODIFY HERE: maximum vocabulary size (default: None)
    ngram_range=(1,2) # MODIFY HERE: min_n, max_n (default: (1,1))
)

The vocabulary size is 5000 tokens
              precision    recall  f1-score   support

    negative       0.68      0.50      0.58        30
    positive       0.91      0.96      0.93       156

    accuracy                           0.88       186
   macro avg       0.80      0.73      0.75       186
weighted avg       0.87      0.88      0.87       186



# Data-Driven Tokenization
We not turn to data-driven tokenization algorithms provided by HuggingFace's `tokenizer` library.

## Data-Driven: Byte-Pair Encoding
The Byte-Pair Encoding (BPE) algorithm is described [here](https://huggingface.co/learn/llm-course/chapter6/5) by HuggingFace. It uses the merging criterion of "the pair that occurs most often" and its token vocabulary is built from the bottom up.

In [70]:
#| code-summary: "MODIFY: Train BPE tokenizer on corpus by vocabulary size, test on example text"
#| code-fold: false
vocab_size = 1000 # MODIFY HERE

tokenizer_bpe = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer_bpe.pre_tokenizer = pre_tokenizers.Whitespace()
trainer_bpe = BpeTrainer(vocab_size=vocab_size, special_tokens=['[UNK]'])
tokenizer_bpe.train_from_iterator(dft['text'], trainer_bpe) # train from corpus

def bpe_tokenize(text):
    return tokenizer_bpe.encode(text).tokens

text = "The San Francisco-based restaurant doesn't charge $10."
print(bpe_tokenize(text))

['The', 'S', 'an', 'F', 'ran', 'c', 'is', 'co', '-', 'b', 'as', 'ed', 'restaurant', 'does', 'n', "'", 't', 'char', 'ge', '[UNK]', '1', '0', '.']


In [71]:
#| code-summary: "BPE: Tokenize dataset and evaluate classifier"
dft['tokens_bpe'] = dft['text'].apply(bpe_tokenize) # entire dataset
sentiment_analysis_pipeline(dft['tokens_bpe'], dft['sentiment']) # Vectorize, classify

The vocabulary size is 925 tokens
              precision    recall  f1-score   support

    negative       0.71      0.40      0.51        30
    positive       0.89      0.97      0.93       156

    accuracy                           0.88       186
   macro avg       0.80      0.68      0.72       186
weighted avg       0.86      0.88      0.86       186



## Data-Driven: WordPiece
The WordPiece algorithm is described [here](https://huggingface.co/learn/llm-course/chapter6/6) by HuggingFace. It uses the merging criterion of "the pair that most improves corpus likelihood" and its token vocabulary is built from the bottom up.

In [72]:
#| code-summary: "Train WordPiece tokenizer with corpus as this dataset, test on example text"
#| code-fold: false
vocab_size = 1000 # MODIFY HERE

tokenizer_wp = Tokenizer(WordPiece(unk_token='[UNK]'))
tokenizer_wp.pre_tokenizer = pre_tokenizers.Whitespace()
trainer_wp = WordPieceTrainer(vocab_size=vocab_size, special_tokens=['[UNK]'])
tokenizer_wp.train_from_iterator(dft['text'], trainer_wp) # train from corpus

def wp_tokenize(text):
    return tokenizer_wp.encode(text).tokens

text = "The San Francisco-based restaurant doesn't charge $10."
print(wp_tokenize(text))

['The', 'S', '##an', 'F', '##ran', '##c', '##is', '##c', '##o', '-', 'b', '##as', '##ed', 'restaurant', 'does', '##n', "'", 't', 'ch', '##arge', '[UNK]', '1', '##0', '.']


In [73]:
#| code-summary: "WordPiece: Tokenize dataset and evaluate classifier"
dft['tokens_wp'] = dft['text'].apply(wp_tokenize)
sentiment_analysis_pipeline(dft['tokens_wp'], dft['sentiment'])

The vocabulary size is 914 tokens
              precision    recall  f1-score   support

    negative       0.83      0.17      0.28        30
    positive       0.86      0.99      0.92       156

    accuracy                           0.86       186
   macro avg       0.85      0.58      0.60       186
weighted avg       0.86      0.86      0.82       186



## Data-Driven: Unigram
The WordPiece algorithm is described [here](https://huggingface.co/learn/llm-course/chapter6/7) by HuggingFace. It uses the merging criterion of "remove the pieces the corpus needs least" and its token vocabulary is pruned from the top down.

In [74]:
#| code-summary: "Train Unigram LM tokenizer with corpus as this dataset, test on example text"
#| code-fold: false
vocab_size = 1000 # MODIFY HERE

tokenizer_uni = Tokenizer(Unigram())
tokenizer_uni.pre_tokenizer = pre_tokenizers.Whitespace()
trainer_uni = UnigramTrainer(vocab_size=vocab_size, unk_token='[UNK]', special_tokens=['[UNK]'])
tokenizer_uni.train_from_iterator(dft['text'], trainer_uni) # train from corpus

def uni_tokenize(text):
    return tokenizer_uni.encode(text).tokens

text = "The San Francisco-based restaurant doesn't charge $10."
print(uni_tokenize(text))

['The', 'S', 'an', 'F', 'r', 'an', 'c', 'is', 'co', '-', 'bas', 'ed', 'restaurant', 'does', 'n', "'", 't', 'charge', '$', '1', '0', '.']


In [75]:
#| code-summary: "Unigram: Tokenize dataset and evaluate classifier"
dft['tokens_uni'] = dft['text'].apply(uni_tokenize)
sentiment_analysis_pipeline(dft['tokens_uni'], dft['sentiment'])

The vocabulary size is 985 tokens
              precision    recall  f1-score   support

    negative       0.71      0.33      0.45        30
    positive       0.88      0.97      0.93       156

    accuracy                           0.87       186
   macro avg       0.80      0.65      0.69       186
weighted avg       0.86      0.87      0.85       186



# Evaluation
What conclusions can we draw about the various tokenization (rules/words vs. data/subwords) and normalization choices and their affect on this dataset's sentiment analysis classification? Can they be generalized across datasets?